# 08 – EC3D Few-Shot Prototypes (NO UNKNOWN)

**Versione paper-aligned**: 11 classi, senza Unknown.

In [ ]:
USE_NO_UNKNOWN = True
N_CLASSES = 11
RESULTS_SUBDIR = 'NO_UNKNOWN'

import sys
from pathlib import Path
import json
import random
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score

ROOT_DIR = Path('..').resolve()
sys.path.insert(0, str(ROOT_DIR))
sys.path.insert(0, str(ROOT_DIR.parent))

from pose_encoder.twostream_stgcn_plus import TwoStreamSTGCNPlusEncoder
from utils import load_ec3d

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42

In [ ]:
DATA_DIR = ROOT_DIR.parent / 'data' / 'EC3D'
LOGS_DIR = ROOT_DIR.parent / 'logs'
RESULTS_DIR = ROOT_DIR.parent / 'results' / 'ec3d' / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ec3d_data = load_ec3d(DATA_DIR, no_unknown=USE_NO_UNKNOWN)
sequences = ec3d_data['sequences']
labels = np.array(ec3d_data['labels'])
train_indices = ec3d_data['train_indices']
test_indices = ec3d_data['test_indices']
NUM_CLASSES = ec3d_data['num_classes']

train_labels = labels[train_indices]
test_labels = labels[test_indices]

In [ ]:
# Load encoder
with open(LOGS_DIR / 'best_epoch.txt') as f:
    BEST_EPOCH = int(f.read().strip())

pose_encoder = TwoStreamSTGCNPlusEncoder(input_dim=3, hidden_channels=[64,128,256,256], output_dim=128, num_nodes=25, dropout=0.1, fusion_dropout=0.3)
pose_encoder.load_state_dict(torch.load(LOGS_DIR / f'pose_encoder_epoch{BEST_EPOCH}.pt', map_location=device))
pose_encoder.to(device).eval()

In [ ]:
def encode_pose(seq, device, max_len=150):
    seq = np.transpose(seq, (0, 2, 1))
    T = seq.shape[0]
    if T < max_len:
        seq = np.concatenate([seq, np.zeros((max_len - T, 25, 3), dtype=np.float32)], axis=0)
    elif T > max_len:
        seq = seq[np.linspace(0, T-1, max_len).astype(int)]
    seq_t = torch.from_numpy(seq.astype(np.float32)).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = pose_encoder(seq_t)
        emb = F.normalize(emb, p=2, dim=1)
    return emb.squeeze(0).cpu().numpy()

train_embeddings = np.vstack([encode_pose(sequences[i], device) for i in tqdm(train_indices, desc='Train')])
test_embeddings = np.vstack([encode_pose(sequences[i], device) for i in tqdm(test_indices, desc='Test')])

In [ ]:
# Few-shot evaluation
K_LIST = [1, 2, 5, 10]
NUM_RUNS = 20

all_results = []

for k in K_LIST:
    print(f'Evaluating k={k}...')
    for run_id in range(NUM_RUNS):
        rng = np.random.default_rng(SEED + run_id * 1000 + k)
        
        # Sample k examples per class and compute prototypes
        prototypes = np.zeros((NUM_CLASSES, 128))
        valid_classes = []
        
        for cid in range(NUM_CLASSES):
            class_indices = np.where(train_labels == cid)[0]
            if len(class_indices) == 0:
                continue
            actual_k = min(k, len(class_indices))
            selected = rng.choice(class_indices, size=actual_k, replace=False)
            proto = train_embeddings[selected].mean(axis=0)
            proto = proto / np.linalg.norm(proto)
            prototypes[cid] = proto
            valid_classes.append(cid)
        
        # Classify test set
        sims = test_embeddings @ prototypes.T
        sims[:, [c for c in range(NUM_CLASSES) if c not in valid_classes]] = -np.inf
        preds = sims.argmax(axis=1)
        
        valid_mask = np.isin(test_labels, valid_classes)
        if valid_mask.sum() > 0:
            acc = accuracy_score(test_labels[valid_mask], preds[valid_mask])
            f1 = f1_score(test_labels[valid_mask], preds[valid_mask], average='macro', zero_division=0)
        else:
            acc, f1 = 0.0, 0.0
        
        all_results.append({'k': k, 'run_id': run_id, 'accuracy': acc, 'macro_f1': f1})

results_df = pd.DataFrame(all_results)

In [ ]:
# Aggregate results
summary = []
for k in K_LIST:
    k_results = results_df[results_df['k'] == k]
    summary.append({
        'k': k,
        'acc_mean': k_results['accuracy'].mean(),
        'acc_std': k_results['accuracy'].std(),
        'f1_mean': k_results['macro_f1'].mean(),
        'f1_std': k_results['macro_f1'].std()
    })
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

In [ ]:
# Save results
results = {
    'experiment': 'few_shot_prototypes', 'version': 'NO_UNKNOWN', 'num_classes': NUM_CLASSES,
    'timestamp': datetime.now().isoformat(),
    'k_list': K_LIST, 'num_runs': NUM_RUNS,
    'summary': summary_df.to_dict('records'),
    'all_runs': results_df.to_dict('records')
}

with open(RESULTS_DIR / 'few_shot_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

summary_df.to_csv(RESULTS_DIR / 'few_shot_summary.csv', index=False)
print(f'Results saved to {RESULTS_DIR}')